In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [3]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv", encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)
data.text

(1000, 2)


0      DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...
1                                               Will do.
2      Nora--Cheryl has emailed dozens of memos about...
3      Dear Sir=2FMadam=2C I know that this proposal ...
4                                                    fyi
                             ...                        
995    So what's the latest? It sounds contradictory ...
996    TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...
997    Barb I will call to explain. Are you back in t...
998      Yang on travelNot free tonite.May work tomorrow
999    sbwhoeopSunday February 21 2010 7:42 PMHShaunH...
Name: text, Length: 1000, dtype: object

### Let's divide the training and test set into two partitions

In [4]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(data, train_size=0.2, random_state=42)

## Data Preprocessing

In [5]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [6]:
import re

def clean_html_code_sentence(html):
    remove_inline = re.sub(r"<script (?:\w+=\".+\")>.+<\/script>|<style>.+<\/style>", "", html)
    remove_comments = re.sub(r"<!--.+-->", "", remove_inline)
    remove_tags = re.sub(r"<\/?\w+>", "", remove_comments)
    return remove_tags

def clean_html_code(data):
    return [clean_html_code_sentence(sentence) for sentence in data]

data_train_clean = clean_html_code(data_train.text)
data_train_clean

["Below is Palau's statement on the recent meeting in the required free association review. The fourth is the keyparagraph. Jake I'll call. Jeff 202-343-2905 oA Compact of Free Association replaced the United States administration of Palau islands which the U.S. took in war andsubsequently administered for the United Nations. The Compact which was signed in 1986 and took effect October 11994 includes provisions of various duration. Some are to last at least 50 years including full U.S. military authority --although the power of the U.S. to deny access to Palau and its extensive waters by the military of any other nation is tocontinue in perpetuity.The amounts and types of U.S. budgetary and programmatic assistance to Palau under the Compact are specified for 15years but the Compact requires reviews of its terms and the entire relationship on its 15th 30th and40th anniversaries. The reviews are to consider the operating requirements of the Government of Palau and the islands'development

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [7]:
def clean_sentence(text):
    remove_prefixed_b = re.sub(r"b'", "", text)
    clean_punctuation = re.sub(f"[{re.escape(string.punctuation)}]", "", remove_prefixed_b)
    only_words = re.sub(r"[^A-Za-z\s]|\s[A-Za-z]\s", "", clean_punctuation)
    trim_spaces_text = re.sub(r"\s(?=\s)", "", only_words)
    return trim_spaces_text.lower()

def clean_data(data):
    return [clean_sentence(sentence) for sentence in data]

data_train_clean_words = clean_data(data_train_clean)
data_train_clean_words

['below is palaus statement on the recent meeting in the required free association review the fourth is the keyparagraph jake ill call jeff oa compact of free association replaced the united states administration of palau islands which the us took in war andsubsequently administered for the united nations the compact which was signed in and took effect october includes provisions of various duration some are to last at least years including full us military authority although the power of the us to deny access to palau and its extensive waters by the military of any other nation is tocontinue in perpetuitythe amounts and types of us budgetary and programmatic assistance to palau under the compact are specified for years but the compact requires reviews of its terms and the entire relationship on its th th andth anniversaries the reviews are to consider the operating requirements of the government of palau and the islandsdevelopment the compact also commits the us to acting on the findi

## Now let's work on removing stopwords
Remove the stopwords.

In [8]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def filter_data(data):
    tokenized_data = [word_tokenize(sentence) for sentence in data]
    stop_words = set(stopwords.words('english'))
    return [[word for word in sentence if word not in stop_words] for sentence in tokenized_data]

filtered_tokens = filter_data(data_train_clean_words)
filtered_tokens

[['palaus',
  'statement',
  'recent',
  'meeting',
  'required',
  'free',
  'association',
  'review',
  'fourth',
  'keyparagraph',
  'jake',
  'ill',
  'call',
  'jeff',
  'oa',
  'compact',
  'free',
  'association',
  'replaced',
  'united',
  'states',
  'administration',
  'palau',
  'islands',
  'us',
  'took',
  'war',
  'andsubsequently',
  'administered',
  'united',
  'nations',
  'compact',
  'signed',
  'took',
  'effect',
  'october',
  'includes',
  'provisions',
  'various',
  'duration',
  'last',
  'least',
  'years',
  'including',
  'full',
  'us',
  'military',
  'authority',
  'although',
  'power',
  'us',
  'deny',
  'access',
  'palau',
  'extensive',
  'waters',
  'military',
  'nation',
  'tocontinue',
  'perpetuitythe',
  'amounts',
  'types',
  'us',
  'budgetary',
  'programmatic',
  'assistance',
  'palau',
  'compact',
  'specified',
  'years',
  'compact',
  'requires',
  'reviews',
  'terms',
  'entire',
  'relationship',
  'th',
  'th',
  'andth',
 

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [9]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_data(data):
    return [[lemmatizer.lemmatize(word) for word in sentence] for sentence in data]

lemmatized_words = lemmatize_data(filtered_tokens)
lemmatized_words

[['palau',
  'statement',
  'recent',
  'meeting',
  'required',
  'free',
  'association',
  'review',
  'fourth',
  'keyparagraph',
  'jake',
  'ill',
  'call',
  'jeff',
  'oa',
  'compact',
  'free',
  'association',
  'replaced',
  'united',
  'state',
  'administration',
  'palau',
  'island',
  'u',
  'took',
  'war',
  'andsubsequently',
  'administered',
  'united',
  'nation',
  'compact',
  'signed',
  'took',
  'effect',
  'october',
  'includes',
  'provision',
  'various',
  'duration',
  'last',
  'least',
  'year',
  'including',
  'full',
  'u',
  'military',
  'authority',
  'although',
  'power',
  'u',
  'deny',
  'access',
  'palau',
  'extensive',
  'water',
  'military',
  'nation',
  'tocontinue',
  'perpetuitythe',
  'amount',
  'type',
  'u',
  'budgetary',
  'programmatic',
  'assistance',
  'palau',
  'compact',
  'specified',
  'year',
  'compact',
  'requires',
  'review',
  'term',
  'entire',
  'relationship',
  'th',
  'th',
  'andth',
  'anniversary',


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [10]:
from collections import Counter

data_val_preprocessing_text = lemmatize_data(filter_data(clean_data(clean_html_code(data_val.text))))
data_train['preprocessed_text'] = [" ".join(sentence) for sentence in lemmatized_words]
data_val['preprocessed_text'] = [" ".join(sentence) for sentence in data_val_preprocessing_text]

ham_tokens = [tokens for tokens, label in zip(lemmatized_words, data_train.label) if label == 0]
spam_tokens = [tokens for tokens, label in zip(lemmatized_words, data_train.label) if label == 1]
ham_words_flat = [word for sentence in ham_tokens for word in sentence]
spam_words_flat = [word for sentence in spam_tokens for word in sentence]
ham_word_counts = Counter(ham_words_flat)
spam_word_counts = Counter(spam_words_flat)

top10_ham = ham_word_counts.most_common(10)
top10_spam = spam_word_counts.most_common(10)
display(top10_ham)
display(top10_spam)

[('pm', 43),
 ('u', 42),
 ('president', 41),
 ('american', 37),
 ('would', 37),
 ('one', 36),
 ('house', 26),
 ('secretary', 25),
 ('mr', 25),
 ('state', 24)]

[('money', 168),
 ('account', 155),
 ('bank', 141),
 ('fund', 113),
 ('u', 94),
 ('million', 82),
 ('business', 76),
 ('contact', 75),
 ('transaction', 74),
 ('transfer', 72)]

## Extra features

In [11]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
556,Below is Palau's statement on the recent meeti...,0,palau statement recent meeting required free a...,0,1,2441
957,Fyi,0,fyi,0,0,3
577,Pis print.From Mills Cheryl D [mailto:MillsCD@...,0,pi printfrom mill cherylmailtomillscdstategovs...,0,0,122
795,Overall the speech is worth reading; if you li...,0,overall speech worth reading like ill put week...,0,0,869
85,We would need Tom and Craig's guidance as to w...,0,would need tom craigs guidance appropriate tim...,0,0,78


## How would work the Bag of Words with Count Vectorizer concept?

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vect = CountVectorizer()
bow_matrix = bow_vect.fit_transform(data_train['preprocessed_text']).toarray()
bow_matrix

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(200, 7479))

In [13]:
bow_df = pd.DataFrame(bow_matrix, columns=bow_vect.get_feature_names_out(), index=data_train['preprocessed_text'])
bow_df['label'] = data_train.label.values
ham_word_counts = bow_df[bow_df['label'] == 0].drop(columns='label').sum().sort_values(ascending=False)
spam_word_counts = bow_df[bow_df['label'] == 1].drop(columns='label').sum().sort_values(ascending=False)
print(ham_word_counts.head(10))
print(spam_word_counts.head(10))

pm           43
president    41
american     37
would        37
one          36
house        26
mr           25
secretary    25
state        24
people       22
dtype: int64
money          168
account        155
bank           141
fund           113
million         82
business        76
contact         75
transaction     74
transfer        72
next            66
dtype: int64


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [14]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

count_vectorizer = CountVectorizer (min_df = 1)
tfidf = TfidfTransformer(norm = "l2")
term_freq_matrix = count_vectorizer.fit_transform(data_train['preprocessed_text'])
tf_idf_matrix = tfidf.fit_transform(term_freq_matrix)

tfidf_df = pd.DataFrame(tf_idf_matrix.toarray(), columns=count_vectorizer.get_feature_names_out())
tfidf_df['label'] = data_train.label.values

ham_word_counts = tfidf_df[tfidf_df['label'] == 0].drop(columns='label').sum().sort_values(ascending=False)
spam_word_counts = tfidf_df[tfidf_df['label'] == 1].drop(columns='label').sum().sort_values(ascending=False)
print(ham_word_counts.head(10))
print(spam_word_counts.head(10))

fyi         12.468328
call         2.611909
talk         2.577152
ok           2.000000
would        1.894232
pm           1.887814
house        1.882098
pls          1.599531
tomorrow     1.583102
like         1.576263
dtype: float64
money          5.006973
bank           4.755091
account        4.593970
fund           3.561560
contact        2.777539
kin            2.726809
next           2.547943
transaction    2.527779
business       2.439639
mr             2.382766
dtype: float64


## And the Train a Classifier?

In [21]:
from textblob.classifiers import NaiveBayesClassifier

label_map = {0: 'ham', 1: 'spam'}

train_data = [(text, label_map[label]) for text, label in zip(data_train['preprocessed_text'], data_train['label'])]
val_data = [(text, label_map[label]) for text, label in zip(data_val['preprocessed_text'], data_val['label'])]

nb_classifier = NaiveBayesClassifier(train_data)
print(data_val['preprocessed_text'][0])
print(f"Predicted type: {nb_classifier.classify(data_val['preprocessed_text'][0])}")
print(f"Real type: {label_map[data_val['label'][0]]}")

dear sir strictlyprivate business proposalam mike chukwu manager bill exchange foreign remittance department zenith international bank plcam writing letter ask support cooperation carry business opportunity department discovered abandoned sum fifteen million united state dollar account belongs one foreign customer died along entire family ofwife two child november inplane crash since heard death expecting nextofkin come put claim money heirbecause release fund account unless someone applies claim nextofkin deceased indicated banking guideline unfortunately neither family member distant relative ever appeared claim said fund upon discoveryi official department agreed make business release total amount account heir fund since one came discovered maintained account bank otherwise fund returned bank treasury unclaimed fund agreed ratio sharing stated thus foreign partner u official department settlement local foreign expences incurred u course business upon successful completion transferan

### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [16]:
# Your code